# WRP Simulation: Lodewijkstraat

This notebook acts as the main execution script. It imports the itinerary generator (`Entity.py`) and the Discrete Event Simulation engine (`DSE.py`), translates the data between them, and runs the required scenarios (Baseline vs +20% capacity).

### Priority Routing Note:
In `DSE.py`, upstream priority is determined by the order in which `.connect()` is called. To ensure `Rest` pulls from `DcDd` before `Hall/Overflow`, we simply execute `dcdd.connect(rest)` *before* `hall_overflow.connect(rest)`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from Entity import CustomerItineraryGenerator
from DSE import (
    Environment, LocationType, QueueLocation, 
    ServiceLocation, VehicleSize, Location, ItineraryItem
)

### 1. Adapter: Translating Entity.py to DSE.py
Your partner's code generates routes as a list of integers (0 to 5). Our `DSE.py` requires `Customer` objects containing `ItineraryItem` objects with `LocationType` Enums. This function bridges that gap.

In [ ]:
def generate_dse_customers(num_customers):
    generator = CustomerItineraryGenerator()
    dse_customers = []
    
    location_map = {
        0: LocationType.MAIN_QUEUE, 1: LocationType.HALL_OVERFLOW, 
        2: LocationType.DCDD, 3: LocationType.GREEN,
        4: LocationType.REST, 5: LocationType.EXIT,
        6: LocationType.HALL_OVERFLOW,
    }

    service_times = {
        LocationType.MAIN_QUEUE: 0, # Main queue doesn't have an inherent service time
        LocationType.HALL_QUEUE: 0,
        LocationType.HALL_OVERFLOW: 240, LocationType.DCDD: 331,
        LocationType.GREEN: 341, LocationType.REST: 141, LocationType.EXIT: 0
    }

    arrival_clock = 0.0 # NEW: Keep track of time

    for _ in range(num_customers):
        raw_itinerary = generator.Next()
        v_size = VehicleSize.BIG if np.random.rand() < 0.47 else VehicleSize.SMALL
        
        dse_itinerary = []
        for i, loc_int in enumerate(raw_itinerary):
            loc_enum = location_map[loc_int]
            item = ItineraryItem(location=loc_enum, service_time=service_times[loc_enum])
            
            # NEW: Set the actual arrival time ONLY for their first stop (Entrance)
            if i == 0:
                item.start_time = arrival_clock

            if i in [1, 2]    
            dse_itinerary.append(item)
            
        import DSE 
        dse_customers.append(DSE.Customer(dse_itinerary, v_size))
        
        # NEW: Space out the next vehicle using an exponential distribution
        # 28800 seconds / 605 cars = ~47.6 sec average gap
        arrival_clock += np.random.exponential(28800 / num_customers) 
        
    return dse_customers

### 2. Network Configuration
We instantiate the exact classes from your `DSE.py` and connect them carefully to establish the correct upstream polling priority.

In [ ]:
def setup_environment(customers):
    # 1. Instantiate Locations
    # Entrance
    entrance = QueueLocation(LocationType.MAIN_QUEUE, maximum_capacity=9999)
    
    # Combined Hall & Overflow: 12 Hall + 10 Overflow = 22 total small bays.
    # Pairs created for big vehicles: [[0, 1], [2, 3], ... [10, 11]]
    hall_pairs = [[i, i+1] for i in range(0, 12, 2)]
    combined_hall = ServiceLocation(LocationType.HALL_QUEUE, max_capacity=22, single_bays=22, single_bay_pairs=hall_pairs)
    
    # DcDd: 7 single bays -> 3 pairs for big vehicles
    dcdd_pairs = [[0, 1], [2, 3], [4, 5]]
    dcdd = ServiceLocation(LocationType.DCDD, max_capacity=7, single_bays=7, single_bay_pairs=dcdd_pairs)
    
    # Green & Rest: 5 single bays each
    green = ServiceLocation(LocationType.GREEN, max_capacity=5, single_bays=5, single_bay_pairs=[])
    rest = ServiceLocation(LocationType.REST, max_capacity=5, single_bays=5, single_bay_pairs=[])
    
    # Exit Sink
    exit_node = Location(LocationType.EXIT, max_capacity=9999)

    # 2. Connect Routing (ORDER DICTATES PRIORITY)
    # Priority for REST: DcDd (Highest) -> Green -> Combined Hall (Lowest)
    dcdd.connect(rest)        # Priority 0
    green.connect(rest)       # Priority 1
    combined_hall.connect(rest) # Priority 2
    
    # Priority for DcDd
    combined_hall.connect(dcdd)
    
    # Priority for Exiting
    rest.connect(exit_node)
    green.connect(exit_node)
    dcdd.connect(exit_node)
    combined_hall.connect(exit_node)
    
    # Main routing from Entrance
    entrance.connect(combined_hall)
    entrance.connect(green)

    # Pack into dictionary for Environment
    locations_dict = {
        LocationType.MAIN_QUEUE: entrance,
        LocationType.HALL_QUEUE: combined_hall,
        LocationType.DCDD: dcdd,
        LocationType.GREEN: green,
        LocationType.REST: rest,
        LocationType.EXIT: exit_node
    }
    
    return Environment(customers, locations_dict, initial_time=0)

### 3. Scenario 1: Baseline (605 Cars per Day)

In [ ]:
print("--- Building Baseline Scenario ---")
baseline_customers = generate_dse_customers(num_customers=605)
baseline_env = setup_environment(baseline_customers)

print("Running Baseline Simulation...")
# Run for roughly an 8 hour day (28,800 seconds)
baseline_env.run(end_time=28800) 
print(f"Simulation finished at time: {baseline_env.time}")

### 4. Scenario 2: 20% Increase in Visitors (726 Cars per Day)

In [ ]:
print("--- Building +20% Scenario ---")
increased_customers = generate_dse_customers(num_customers=726)
increased_env = setup_environment(increased_customers)

print("Running +20% Simulation...")
increased_env.run(end_time=28800)
print(f"Simulation finished at time: {increased_env.time}")

print("\nDone! You can now iterate over increased_customers to extract wait times for analysis.")

In [ ]:
increased_customers

## 📊 Simulation Analytics & Visualization

In this section, we parse the raw object traces into a `pandas.DataFrame` to analyze queue dynamics, pinpoint bottlenecks, and visualize the system's performance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean plotting style
sns.set_theme(style="whitegrid")

def extract_traces_to_df(customers):
    """Converts a list of Customer objects into a flat pandas DataFrame."""
    data = []
    for i, c in enumerate(customers):
        # Fallback ID if your customer class doesn't have an explicit 'id' attribute yet
        cid = getattr(c, 'id', i) 
        
        for item in c.itinerary:
            data.append({
                'customer_id': cid,
                'vehicle_size': c.vehicle_size.name,
                'location': item.location.name,
                'start_time': item.start_time,
                'end_time': item.end_time,
                'wait_time': item.time_waiting,
                'service_time': item.service_time
            })
            
    df = pd.DataFrame(data)
    
    # Calculate when the customer officially left the queue and began service
    df['queue_exit_time'] = df['start_time'] + df['wait_time']
    
    return df

# Create DataFrames for both scenarios
df_base = extract_traces_to_df(baseline_customers)
df_inc = extract_traces_to_df(increased_customers)

# Display the first few rows of the baseline to verify
display(df_base)

### 1. Identifying Blockage Points (Average Wait Times)
The easiest way to find a bottleneck is to see where vehicles spend the most time waiting.

In [ ]:
def plot_wait_times(df, title):
    # Exclude the EXIT node since wait time is always 0 there
    df_filtered = df[df['location'] != 'EXIT']
    
    wait_summary = df_filtered.groupby('location')['wait_time'].mean().reset_index()
    wait_summary = wait_summary.sort_values(by='wait_time', ascending=False)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=wait_summary, x='location', y='wait_time', palette='rocket')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('Average Wait Time (seconds)', fontsize=12)
    plt.xlabel('Location', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_wait_times(df_base, 'Baseline: Average Wait Time by Location (Blockage Points)')
plot_wait_times(df_inc, '+20% Scenario: Average Wait Time by Location')

### 2. Queue Length Over Time (Step Plot)
To calculate exact queue length at any given second, we generate an "event timeline" (+1 when someone arrives, -1 when they leave the queue) and calculate the cumulative sum.

In [ ]:
def plot_queue_lengths(df, title):
    locations = [loc for loc in df['location'].unique() if loc != 'EXIT']
    
    plt.figure(figsize=(14, 7))
    
    for loc in locations:
        loc_df = df[df['location'] == loc]
        
        # +1 when entering the queue (start_time)
        arrivals = pd.DataFrame({'time': loc_df['start_time'], 'change': 1})
        # -1 when leaving the queue to start service (queue_exit_time)
        departures = pd.DataFrame({'time': loc_df['queue_exit_time'], 'change': -1})
        
        # Combine, sort chronologically, and calculate cumulative sum
        events = pd.concat([arrivals, departures]).sort_values(by=['time', 'change'], ascending=[True, False])
        events['queue_length'] = events['change'].cumsum()
        
        # Use a step plot for accurate queue visualization
        plt.step(events['time'], events['queue_length'], where='post', label=loc, alpha=0.8, linewidth=2)
        
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Simulation Time (seconds)', fontsize=12)
    plt.ylabel('Number of Vehicles in Queue', fontsize=12)
    plt.legend(title='Location')
    plt.tight_layout()
    plt.show()

plot_queue_lengths(df_base, 'Baseline: Queue Length Dynamics Over Time')
# plot_queue_lengths(df_inc, '+20% Scenario: Queue Length Dynamics Over Time')

### 3. Total System Turnaround Time
How long does a customer spend in the WRP from the moment they arrive at the Entrance to the moment they Exit?

In [ ]:
def plot_turnaround_time(df, title):
    # Get the min entry time and max exit time for each customer
    sys_time = df.groupby('customer_id').agg(
        entry_time=('start_time', 'min'),
        exit_time=('end_time', 'max'),
        vehicle_size=('vehicle_size', 'first') # Keep vehicle size for hue plotting
    )
    
    sys_time['total_time'] = sys_time['exit_time'] - sys_time['entry_time']
    
    plt.figure(figsize=(10, 6))
    sns.histplot(data=sys_time, x='total_time', hue='vehicle_size', bins=40, kde=True, palette='mako')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Total Time in System (seconds)', fontsize=12)
    plt.ylabel('Customer Count', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print(f"Average Total Time in System: {sys_time['total_time'].mean():.2f} seconds")
    print(f"Maximum Total Time in System: {sys_time['total_time'].max():.2f} seconds")

plot_turnaround_time(df_base, 'Baseline: Distribution of Total Turnaround Time')


In [ ]:
import plotly.express as px
import pandas as pd

def build_debugging_visualizer(df, scenario_name="Simulation"):
    """
    Transforms the simulation log into an interactive HTML Gantt chart.
    Splits 'Waiting' and 'Servicing' into distinct visual blocks.
    """
    # 1. We need to split each customer's location visit into two distinct events:
    # Event A: Waiting in line
    # Event B: Being serviced
    viz_data = []
    
    # Base dummy date required by Plotly for timelines
    base_time = pd.Timestamp('2024-01-01 08:00:00') 
    
    for _, row in df.iterrows():
        if row['location'] == 'EXIT':
            continue # Skip exit node for visualization
            
        start_t = row['start_time']
        queue_exit_t = row['start_time'] + row['wait_time']
        end_t = row['end_time']
        
        # Add 'Waiting' block if they actually waited
        if row['wait_time'] > 0:
            viz_data.append({
                'Customer': f"Cust {row['customer_id']} ({row['vehicle_size']})",
                'State': f"WAITING: {row['location']}",
                'Location': row['location'],
                'Start': base_time + pd.to_timedelta(start_t, unit='s'),
                'Finish': base_time + pd.to_timedelta(queue_exit_t, unit='s'),
                'Duration (s)': row['wait_time'],
                'Type': 'Waiting'
            })
            
        # Add 'Servicing' block
        if row['service_time'] > 0:
            viz_data.append({
                'Customer': f"Cust {row['customer_id']} ({row['vehicle_size']})",
                'State': f"SERVICING: {row['location']}",
                'Location': row['location'],
                'Start': base_time + pd.to_timedelta(queue_exit_t, unit='s'),
                'Finish': base_time + pd.to_timedelta(end_t, unit='s'),
                'Duration (s)': row['service_time'],
                'Type': 'Servicing'
            })

    viz_df = pd.DataFrame(viz_data)
    
    if viz_df.empty:
        print("No data to visualize.")
        return

    # 2. Build the Plotly Gantt Chart
    fig = px.timeline(
        viz_df, 
        x_start="Start", 
        x_end="Finish", 
        y="Customer", 
        color="State",
        hover_data=["Location", "Type", "Duration (s)"],
        title=f"{scenario_name}: Customer Logic Flow Debugger"
    )
    
    # 3. Clean up the layout
    fig.update_yaxes(autorange="reversed") # Put Customer 0 at the top
    fig.update_layout(
        xaxis_title="Simulation Time",
        height=max(400, len(df['customer_id'].unique()) * 30), # Auto-scale height based on customer count
        showlegend=True,
        hovermode="closest"
    )
    
    # Show in notebook
    fig.show()
    
    # Optional: Save as a standalone HTML file you can open in Chrome
    # fig.write_html(f"{scenario_name.replace(' ', '_')}_debugger.html")

# Call it using your first 10 customers so the chart isn't too crowded while debugging!
debug_df = df_base[df_base['customer_id'] < 10]
build_debugging_visualizer(debug_df, "Baseline First 10 Customers")

In [ ]:
import json

def export_for_animator(df, filename="sim_trace.json"):
    # FIX: Exclude the exit node AND exclude incomplete events (end_time == 0)
    df_anim = df[(df['location'] != 'EXIT') & (df['end_time'] > 0)].copy()
    
    customers_data = []
    
    for cid, group in df_anim.groupby('customer_id'):
        # Get vehicle size from the first row of this customer
        v_size = group['vehicle_size'].iloc[0] 
        
        path = []
        for _, row in group.iterrows():
            path.append({
                "location": row['location'],
                # "bay": row.get('bay_number', 0), # Uncomment if you add bays!
                "arrive_queue": row['start_time'],
                "start_service": row['queue_exit_time'],
                "finish_service": row['end_time']
            })
            
        customers_data.append({
            "id": int(cid),
            "size": v_size,
            "path": path
        })
        
    with open(filename, 'w') as f:
        json.dump(customers_data, f, indent=2)
        
    print(f"Exported {len(customers_data)} customers to {filename}")

# Run it
export_for_animator(df_base, "baseline_anim.json")
